In [29]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

In [30]:
DATA_DIR = '../data/raw/'

train         = pd.read_csv(DATA_DIR + 'application_train.csv')
test          = pd.read_csv(DATA_DIR + 'application_test.csv')
bureau        = pd.read_csv(DATA_DIR + 'bureau.csv')
bureau_bal    = pd.read_csv(DATA_DIR + 'bureau_balance.csv')
prev_app      = pd.read_csv(DATA_DIR + 'previous_application.csv')
pos_cash      = pd.read_csv(DATA_DIR + 'POS_CASH_balance.csv')
installments  = pd.read_csv(DATA_DIR + 'installments_payments.csv')
credit_card   = pd.read_csv(DATA_DIR + 'credit_card_balance.csv')

In [28]:
tables = {
    'application_train':    train,
    'application_test':     test,
    'bureau':               bureau,
    'bureau_balance':       bureau_bal,
    'previous_application': prev_app,
    'pos_cash_balance':     pos_cash,
    'installments_payments':installments,
    'credit_card_balance':  credit_card,
}

total_memory_mb = 0
for name, df in tables.items():
    memory_mb = df.memory_usage(deep = True).sum() / 1024**2
    total_memory_mb = total_memory_mb + memory_mb
    print(f'{name:<30} {df.shape[0]:>10,} rows  {df.shape[1]:>4} cols  {memory_mb:>8,.1f} MB')
print(f'Total Memory in GB: {total_memory_mb/1024:2.1f} GB')

application_train                 307,511 rows   122 cols     536.7 MB
application_test                   48,744 rows   121 cols      84.7 MB
bureau                          1,716,428 rows    17 cols     512.1 MB
bureau_balance                 27,299,925 rows     3 cols   1,926.6 MB
previous_application            1,670,214 rows    37 cols   1,900.6 MB
pos_cash_balance               10,001,358 rows     8 cols   1,137.3 MB
installments_payments          13,605,401 rows     8 cols     830.4 MB
credit_card_balance             3,840,312 rows    23 cols     875.7 MB
Total Memory in GB: 7.6 GB


A few things to note:
* ~7.6 GB total memory across the table, that's a lot to hold in RAM at once.
* 

In [25]:
print(df.memory_usage(deep = True).sum() / 1024**2)

875.6876678466797


In [21]:
# Find all columns that look like ID columns across all tables
all_id_cols = set()

for name, df in tables.items():
    id_like = [col for col in df.columns if 'SK_ID' in col]
    all_id_cols.update(id_like)

print('All ID-like columns found across tables:')
for col in sorted(all_id_cols):
    appears_in = [name for name, df in tables.items() if col in df.columns]
    print(f'  {col:<20} → {appears_in}')

All ID-like columns found across tables:
  SK_ID_BUREAU         → ['bureau', 'bureau_balance']
  SK_ID_CURR           → ['application_train', 'application_test', 'bureau', 'previous_application', 'pos_cash_balance', 'installments_payments', 'credit_card_balance']
  SK_ID_PREV           → ['previous_application', 'pos_cash_balance', 'installments_payments', 'credit_card_balance']


In [22]:
id_cols = ['SK_ID_CURR', 'SK_ID_BUREAU', 'SK_ID_PREV']

for col in id_cols:
    print(f'\n── {col} ──')
    for name, df in tables.items():
        if col in df.columns:
            print(f'  {name:<30} dtype: {df[col].dtype}   min: {df[col].min():,}   max: {df[col].max():,}')


── SK_ID_CURR ──
  application_train              dtype: int64   min: 100,002   max: 456,255
  application_test               dtype: int64   min: 100,001   max: 456,250
  bureau                         dtype: int64   min: 100,001   max: 456,255
  previous_application           dtype: int64   min: 100,001   max: 456,255
  pos_cash_balance               dtype: int64   min: 100,001   max: 456,255
  installments_payments          dtype: int64   min: 100,001   max: 456,255
  credit_card_balance            dtype: int64   min: 100,006   max: 456,250

── SK_ID_BUREAU ──
  bureau                         dtype: int64   min: 5,000,000   max: 6,843,457
  bureau_balance                 dtype: int64   min: 5,001,709   max: 6,842,888

── SK_ID_PREV ──
  previous_application           dtype: int64   min: 1,000,001   max: 2,845,382
  pos_cash_balance               dtype: int64   min: 1,000,001   max: 2,843,499
  installments_payments          dtype: int64   min: 1,000,001   max: 2,843,499
  credit_car

## ID Column Analysis

All three ID columns (`SK_ID_CURR`, `SK_ID_BUREAU`, `SK_ID_PREV`) are inferred as `int64` by default.  
Max values are `456k`, `6.8M`, and `2.8M` respectively — all well within `int32` range (max ~2.1B).  
We'll cast these to `int32` in the loader to cut memory on these columns by half.

In [33]:
# Dtype breakdown per table
for name, df in tables.items():
    print(f'\n{name}')
    print(f'  {"dtype":<12} {"count":>6}')
    print(f'  {"-"*20}')
    for dtype, count in df.dtypes.value_counts().items():
        print(f'  {str(dtype):<12} {int(count):>6}')


application_train
  dtype         count
  --------------------
  float64          65
  int64            41
  str              16

application_test
  dtype         count
  --------------------
  float64          65
  int64            40
  str              16

bureau
  dtype         count
  --------------------
  float64           8
  int64             6
  str               3

bureau_balance
  dtype         count
  --------------------
  int64             2
  str               1

previous_application
  dtype         count
  --------------------
  str              16
  float64          15
  int64             6

pos_cash_balance
  dtype         count
  --------------------
  int64             5
  float64           2
  str               1

installments_payments
  dtype         count
  --------------------
  float64           5
  int64             3

credit_card_balance
  dtype         count
  --------------------
  float64          15
  int64             7
  str               1


Too many `float64` columns in application_train and application_test, this may be happening because pandas cannot store `NaN` in an integer column

In [34]:
# Float64 columns that only contain whole numbers (likely int + NaN)
print('Columns read as float64 but contain only whole numbers:\n')

for name, df in tables.items():
    float_cols = df.select_dtypes(include='float64').columns
    disguised = [
        col for col in float_cols
        if df[col].dropna().apply(float.is_integer).all()
    ]
    if disguised:
        print(f'{name} ({len(disguised)}):')
        for col in disguised:
            null_pct = df[col].isnull().mean() * 100
            print(f'  {col:<45} null: {null_pct:.1f}%')
        print()

Columns read as float64 but contain only whole numbers:

application_train (13):
  OWN_CAR_AGE                                   null: 66.0%
  CNT_FAM_MEMBERS                               null: 0.0%
  OBS_30_CNT_SOCIAL_CIRCLE                      null: 0.3%
  DEF_30_CNT_SOCIAL_CIRCLE                      null: 0.3%
  OBS_60_CNT_SOCIAL_CIRCLE                      null: 0.3%
  DEF_60_CNT_SOCIAL_CIRCLE                      null: 0.3%
  DAYS_LAST_PHONE_CHANGE                        null: 0.0%
  AMT_REQ_CREDIT_BUREAU_HOUR                    null: 13.5%
  AMT_REQ_CREDIT_BUREAU_DAY                     null: 13.5%
  AMT_REQ_CREDIT_BUREAU_WEEK                    null: 13.5%
  AMT_REQ_CREDIT_BUREAU_MON                     null: 13.5%
  AMT_REQ_CREDIT_BUREAU_QRT                     null: 13.5%
  AMT_REQ_CREDIT_BUREAU_YEAR                    null: 13.5%

application_test (14):
  DAYS_REGISTRATION                             null: 0.0%
  OWN_CAR_AGE                                   null: 66.3%
  

## Float64 disguised integers

13 columns in application_train are whole numbers stored as float64 — pandas
does this automatically when a column has NaN but the dtype would otherwise
be integer. 

We're leaving these as float64 in the loader. Deciding what to do with the
nulls (impute, flag, drop) is a feature engineering decision that comes after
we understand what the missingness means. Converting them now would be
premature.

Only the ID columns get explicit dtype overrides in the loader — their ranges
are confirmed safe for int32 and they have no nulls or ambiguity.